# DUNE/FABRIC incremental replay viewer

Run this cell before starting receiver capture. It watches the atomic compressed snapshot copied to LoomAI and redraws only when the detector sample count increases.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

SNAPSHOT = Path("/tmp/replay-live-current.csv.gz")
EXPECTED_SAMPLES = 409_600
history = []
last_sample_count = -1

for update in range(180):
    try:
        if not SNAPSHOT.exists():
            raise FileNotFoundError("Waiting for the first snapshot")

        data = pd.read_csv(
            SNAPSHOT,
            compression="gzip",
            usecols=["tick", "adc_value", "wire_id", "tpc", "plane"],
            on_bad_lines="skip",
        ).dropna()
        data = data[(data["tpc"] == 0) & (data["plane"] == 0)]
        sample_count = len(data)

        if sample_count == last_sample_count:
            time.sleep(5)
            continue
        last_sample_count = sample_count

        pivot = (
            data.groupby(["wire_id", "tick"])["adc_value"]
            .max()
            .unstack()
            .reindex(index=range(200), columns=range(2048))
        )
        values = pivot.to_numpy(dtype=float, copy=True)
        for row in range(values.shape[0]):
            valid = np.isfinite(values[row])
            if valid.any():
                values[row, valid] -= np.median(values[row, valid])

        completion = min(100, sample_count / EXPECTED_SAMPLES * 100)
        figure = go.Figure(go.Heatmap(
            z=values,
            x=pivot.columns,
            y=pivot.index,
            colorscale="Thermal",
            zmin=-20,
            zmax=50,
            colorbar={"title": "ADC - pedestal"},
        ))
        figure.update_layout(
            title=(
                f"DUNE/FABRIC replay - {sample_count:,} samples "
                f"({completion:.1f}%)"
            ),
            xaxis_title="Time tick",
            yaxis_title="Wire ID",
            template="plotly_dark",
            height=750,
        )
        figure.update_xaxes(range=[0, 2047])
        figure.update_yaxes(range=[199, 0])

        history.append({
            "time": time.strftime("%H:%M:%S"),
            "samples": sample_count,
            "completion_%": round(completion, 1),
            "wires_seen": int(data["wire_id"].nunique()),
            "max_tick": int(data["tick"].max()),
        })
        clear_output(wait=True)
        display(figure)
        display(pd.DataFrame(history))

        if sample_count >= EXPECTED_SAMPLES:
            print("Replay complete.")
            break
    except Exception as error:
        clear_output(wait=True)
        print("Waiting for replay data:", error)

    time.sleep(5)
